In [1]:
import os
data_dir = '/home/devang/Projects/PilotCrew/Data-Science-Bench/tasks/task_10/data'
print(os.listdir(data_dir))

['depots.csv', 'shipments.csv', 'scans.csv']


In [2]:
import pandas as pd
depots = pd.read_csv(os.path.join(data_dir, 'depots.csv'))
shipments = pd.read_csv(os.path.join(data_dir, 'shipments.csv'))
scans = pd.read_csv(os.path.join(data_dir, 'scans.csv'))

print("Depots columns:", depots.columns)
print("Shipments columns:", shipments.columns)
print("Scans columns:", scans.columns)


Depots columns: Index(['depot_id', 'region'], dtype='str')
Shipments columns: Index(['shipment_id', 'ship_date', 'origin_depot', 'dest_depot', 'carrier',
       'weight_kg', 'promised_days'],
      dtype='str')
Scans columns: Index(['shipment_id', 'scan_time', 'status'], dtype='str')


In [3]:
print(depots.head())
print(shipments.head())
print(scans.head())
print(scans['status'].unique())

  depot_id   region
0      D01     West
1      D02     East
2      D03    South
3      D04    North
4      D05  Central
  shipment_id   ship_date origin_depot dest_depot carrier  weight_kg  \
0     SHP0001  2024-08-01          D03        D05     CR2         12   
1     SHP0002  2024-08-02          D05        D02     CR3         19   
2     SHP0003  2024-08-03          D01        D05     CR1         26   
3     SHP0004  2024-08-04          D03        D02     CR2         33   
4     SHP0005  2024-08-05          D05        D06     CR3          9   

   promised_days  
0              3  
1              1  
2              3  
3              1  
4              3  
  shipment_id            scan_time        status
0     SHP0001  2024-08-01 09:00:00     picked_up
1     SHP0001  2024-08-01 16:00:00  hub_received
2     SHP0001  2024-08-02 06:00:00    in_transit
3     SHP0001  2024-08-02 19:00:00    in_transit
4     SHP0001  2024-08-03 03:00:00     delivered
<StringArray>
['picked_up', 'hub_receiv

In [4]:
# 1. Filter shipments originating from South-region depots
south_depots = depots[depots['region'] == 'South']['depot_id']
south_shipments = shipments[shipments['origin_depot'].isin(south_depots)]

# 2. Get scans for these shipments
south_scans = scans[scans['shipment_id'].isin(south_shipments['shipment_id'])]

# 3. Find pickup and first hub receipt times
south_scans['scan_time'] = pd.to_datetime(south_scans['scan_time'])

pickups = south_scans[south_scans['status'] == 'picked_up'].groupby('shipment_id')['scan_time'].min().reset_index()
pickups.rename(columns={'scan_time': 'pickup_time'}, inplace=True)

hub_receipts = south_scans[south_scans['status'] == 'hub_received'].groupby('shipment_id')['scan_time'].min().reset_index()
hub_receipts.rename(columns={'scan_time': 'hub_receipt_time'}, inplace=True)

# 4. Merge and calculate time difference
merged = pd.merge(pickups, hub_receipts, on='shipment_id')
merged['time_diff_hours'] = (merged['hub_receipt_time'] - merged['pickup_time']).dt.total_seconds() / 3600

# 5. Calculate median
median_time = merged['time_diff_hours'].median()
print(f"Median time from pickup to first hub receipt: {median_time} hours")

Median time from pickup to first hub receipt: 12.5 hours


In [5]:
# Let's double check the logic
south_depots = depots[depots['region'] == 'South']
print(south_depots)

south_shipments = shipments[shipments['origin_depot'].isin(south_depots['depot_id'])]
print(south_shipments.head())

south_scans = scans[scans['shipment_id'].isin(south_shipments['shipment_id'])].copy()
south_scans['scan_time'] = pd.to_datetime(south_scans['scan_time'])

pickups = south_scans[south_scans['status'] == 'picked_up'].groupby('shipment_id')['scan_time'].min().reset_index()
hub_receipts = south_scans[south_scans['status'] == 'hub_received'].groupby('shipment_id')['scan_time'].min().reset_index()

merged = pd.merge(pickups, hub_receipts, on='shipment_id', suffixes=('_pickup', '_hub'))
merged['diff_hours'] = (merged['scan_time_hub'] - merged['scan_time_pickup']).dt.total_seconds() / 3600
print(merged['diff_hours'].median())

  depot_id region
2      D03  South
   shipment_id   ship_date origin_depot dest_depot carrier  weight_kg  \
0      SHP0001  2024-08-01          D03        D05     CR2         12   
3      SHP0004  2024-08-04          D03        D02     CR2         33   
6      SHP0007  2024-08-07          D03        D05     CR2         23   
9      SHP0010  2024-08-10          D03        D02     CR2         13   
12     SHP0013  2024-08-13          D03        D05     CR2         34   

    promised_days  
0               3  
3               1  
6               3  
9               1  
12              3  
12.5
